In [1]:
%load_ext autoreload
%autoreload 2

## 1. Imports

In [2]:
import os
import sys

sys.path.append("..")

import random

import numpy as np
import torch
import torch.distributions as TD
from tqdm import tqdm

import wandb
from src.models.energy_based import EGEOT
from src.samplers.energy_based.sample_buffer import SampleBufferEgEOT
from src.samplers.from_dataset import DatasetSampler
from src.utils.train import compute_loss, update_average
from src.plotting.distributions import plot_images

In [3]:
device = torch.device(f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [4]:
torch.set_default_device(device)
# dtype = torch.float64
# torch.torch.set_default_dtype(dtype)

## 2. Config

In [5]:
from configs.energy_based.cost import MLPCostConfig, MLPLSECostConfig, MLPL2CostConfig
from configs.energy_based.dataset import DatasetConfig, MiniBatchConfig
from configs.energy_based.model import EBMConfig
from configs.energy_based.optimizer import OptPairedConfig, OptUnpairedConfig
from configs.energy_based.potential import PotentialConfig
from configs.energy_based.sampling import LangevinConfig
from configs.energy_based.train import TrainConfig

In [6]:
Q_X_UNPAIRED_SAMPLES = 6990
R_Y_UNPAIRED_SAMPLES = 7141
P_XY_PAIRED_SAMPLES = 128
LR_PAIRED = 1e-5
LR_UNPAIRED = 1e-5
SAMPLING_NUM_ITER = 100
MAX_STEPS = 1000
PAIRED_BATCH_SIZE = 1024
UNPAIRED_BATCH_SIZE = 1024
PLOT_EVERY = 10

In [7]:
NUM_LABELED = 10
DATASET = 'fmnist2mnist'
COST = 'Weak_Energy'
DATASET_PATH = '../datasets/'

BATCH_SIZE = 256
C_SIZE = 1
Z_SIZE = 2
IMG_SIZE = 32
T_ITERS = 10
D_LR = 1e-5 
T_LR = 1e-5
NC = 1
ZD = 128
Z_STD = 1.
PLOT_INTERVAL = 500
CPKT_INTERVAL = 10000
MAX_STEPS = 2501 #For illustration purpose only. For training use 60001
SEED = 0x000001

In [8]:
dataset_config = DatasetConfig(
    P_XY_paired=P_XY_PAIRED_SAMPLES, Q_X_unpaired=Q_X_UNPAIRED_SAMPLES, R_Y_unpaired=R_Y_UNPAIRED_SAMPLES
)
model_config = EBMConfig(sampling=LangevinConfig(num_iterations=SAMPLING_NUM_ITER))

opt_unpaired_config = OptUnpairedConfig(lr=LR_UNPAIRED)
opt_paired_config = OptPairedConfig(lr=LR_PAIRED)

train_config = TrainConfig(
    steps_to=MAX_STEPS, paired_batch_size=PAIRED_BATCH_SIZE, unpaired_batch_size=UNPAIRED_BATCH_SIZE, plot_every=PLOT_EVERY,
)

In [9]:
torch.manual_seed(train_config.seed)
np.random.seed(train_config.seed)
random.seed(train_config.seed)

## 3. Create data and samplers

In [10]:
from torchvision.transforms import Compose, Resize, Normalize, ToTensor, Lambda
import torchvision.datasets as datasets

In [11]:
source_subset = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7 ,8, 9])
new_labels_source = {0:0, 1:1, 2:2, 3:3, 4:4, 5:5, 6:6, 7:7, 8:8, 9:9}
target_subset = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
new_labels_target = {0:0, 1:1, 2:2, 3:3, 4:4, 5:5, 6:6, 7:7, 8:8, 9:9}
scheduler_milestones=[10000, 20000, 30000, 40000, 50000]

source_transform = Compose([
    Resize((IMG_SIZE, IMG_SIZE)), 
    ToTensor(),
    Normalize((0.5), (0.5)),
])
target_transform = source_transform

if DATASET == 'mnist2kmnist':
    source = datasets.MNIST
    target = datasets.KMNIST
    
elif DATASET == 'fmnist2mnist':
    source = datasets.FashionMNIST
    target = datasets.MNIST
    
elif DATASET == 'mnist2usps':
    source = datasets.MNIST
    target = datasets.USPS
    
elif DATASET == 'mnist2mnistm':
    source = datasets.MNIST
    target = MNISTM
    NC = 3
    source_transform = Compose([
        Resize((IMG_SIZE, IMG_SIZE)), 
        ToTensor(),
        Normalize((0.5), (0.5)), 
        Lambda(lambda x: -x.repeat(3,1,1))])
    target_transform = Compose([
        Resize(IMG_SIZE),
        ToTensor(),
        Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
    
OUTPUT_PATH = '../saved_models/{}/'.format(DATASET)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)
    
torch.manual_seed(SEED); np.random.seed(SEED)

In [12]:
from src.samplers.guided_old import PairedSubsetSampler, SubsetGuidedDataset, get_indicies_subset

In [13]:
source_train = source(root=DATASET_PATH, train=True, download=True, transform=source_transform)
subset_samples, labels, source_class_indicies = get_indicies_subset(source_train, 
                                                                    new_labels = new_labels_source, 
                                                                    classes=len(source_subset), 
                                                                    subset_classes=source_subset)
source_train =  torch.utils.data.TensorDataset(torch.stack(subset_samples), 
                                               torch.LongTensor(labels))


target_train = target(root=DATASET_PATH, train=True, download=True, transform=target_transform)   
target_subset_samples, target_labels, target_class_indicies = get_indicies_subset(target_train, 
                                                                                  new_labels = new_labels_target, 
                                                                                  classes=len(target_subset), 
                                                                                  subset_classes=target_subset)
target_train = torch.utils.data.TensorDataset(torch.stack(target_subset_samples), 
                                              torch.LongTensor(target_labels))

train_set = SubsetGuidedDataset(source_train, target_train, 
                                num_labeled=NUM_LABELED, 
                                in_indicies = source_class_indicies, 
                                out_indicies = target_class_indicies)

full_set = SubsetGuidedDataset(source_train, target_train, 
                               num_labeled='all', 
                               in_indicies = source_class_indicies, 
                               out_indicies = target_class_indicies)

T_XY_sampler = PairedSubsetSampler(train_set, subsetsize=C_SIZE)
D_XY_sampler = PairedSubsetSampler(full_set, subsetsize=1)
X_fixed, Y_fixed = D_XY_sampler.sample(10)

## 4. Model initialization

In [13]:
from src.costs.convolutional import NonlocalCost, ResNetCost, UnetV2Cost, UnetCost
from src.potentials.convolutional import NonlocalPotential, ResNetPotential

In [14]:
# potential = NonlocalPotential(n_c=NC, n_f=IMG_SIZE)
potential = ResNetPotential(IMG_SIZE, nc=NC).to(device) 
# potential = UnetPotential(NC, NC, ZD, base_factor=48).cuda()

In [15]:
from src.auxiliary_models.unet import CondUNetV2

In [16]:
# cost = NonlocalCost(n_c=NC, n_f=IMG_SIZE)
# cost = ResNetCost(IMG_SIZE, nc=NC).to(device)

# cost = UnetV2Cost(NC).to(device)

In [17]:
# TODO: add to config
BASIC_NOISE_VAR = 1.0
P_SAMPLE_BUFFER_REPLAY = 0.95
SAMPLE_BUFFER_SAMPLES = 10000

In [18]:
X, Y = T_XY_sampler.sample(10)

NameError: name 'T_XY_sampler' is not defined

In [19]:
X.squeeze(1).shape

NameError: name 'X' is not defined

In [ ]:
X, Y = T_XY_sampler.sample(1)
X = X.squeeze(1)
Y = Y.squeeze(1)

In [20]:
X = torch.randn((10, 1, 32, 32))
Y = torch.randn((10, 1, 32, 32))

In [21]:
Y.repeat((1, 8, 1, 1)).shape

torch.Size([10, 8, 32, 32])

In [23]:
net = CondUNetV2(1, 1, 1, 1)
net(X, Y)

RuntimeError: The size of tensor a (2) must match the size of tensor b (32) at non-singleton dimension 3

RuntimeError: The size of tensor a (2) must match the size of tensor b (32) at non-singleton dimension 3

In [22]:
basic_noise_gen = TD.Normal(torch.zeros_like(X[0]).to(device), torch.ones_like(X[0]).to(device) * BASIC_NOISE_VAR)

sample_buffer_instance = SampleBufferEgEOT(
    basic_noise_gen, p=P_SAMPLE_BUFFER_REPLAY, max_samples=SAMPLE_BUFFER_SAMPLES, device=device
)

In [23]:
model = EGEOT(potential, cost, sample_buffer_instance, model_config)

In [24]:
# For EMA update
if train_config.ema_update:
    model_copy = EGEOT(potential, cost, sample_buffer_instance, model_config)

## 5. Optimizers initialization

In [26]:
D_opt_unpaired = torch.optim.Adam(model.potential.parameters(), **opt_unpaired_config.model_dump())

In [27]:
D_opt_paired = torch.optim.Adam(model.cost.parameters(), **opt_paired_config.model_dump())

In [30]:
# TODO: refactor this config
EXP_NAME = (
    "EgEOT_fmnit2mnist_"
    + "_UnetCost_"
    + f"P_XY_PAIRED_{dataset_config.P_XY_paired}_"
    + f"Q_X_UNPAIRED_{dataset_config.Q_X_unpaired}_"
    + f"R_Y_UNPAIRED_{dataset_config.R_Y_unpaired}_"
    + f"LR_PAIRED_{opt_paired_config.lr}_"
    + f"LR_UNPAIRED_{opt_unpaired_config.lr}_"
    + f"SAMPLING_STEPS_{model_config.sampling.num_iterations}_"
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    X_DIM=dataset_config.x_dim,
    Y_DIM=dataset_config.y_dim,
    D_LR_PAIRED=opt_paired_config.lr,
    D_LR_UNPAIRED=opt_unpaired_config.lr,
    BATCH_SIZE=train_config.unpaired_batch_size,
    P_XY_PAIRED_SAMPLES=dataset_config.P_XY_paired,
    Q_X_UNPAIRED_SAMPLES=dataset_config.Q_X_unpaired,
    R_Y_UNPAIRED_SAMPLES=dataset_config.R_Y_unpaired,
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH, exist_ok=True)

In [31]:
if train_config.steps_from > 0:
    D_opt_unpaired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{train_config.steps_from}.pt")))
    D_opt_paired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_paired_{train_config.steps_from}.pt")))

## 6. Model training

In [32]:
wandb.init(name=EXP_NAME, config=config)

for step in tqdm(range(train_config.steps_from, train_config.steps_to)):
    # training loop
    D_opt_unpaired.zero_grad()

    X, Y = T_XY_sampler.sample(BATCH_SIZE)
    X = X.squeeze(1)
    Y = Y.squeeze(1)
    
    output_unpaired = model.compute_unpaired_loss(X, Y, compute_stats=True)
    D_loss_unpaired = output_unpaired["loss"]

    wandb.log({f"Unpaired: Loss": D_loss_unpaired.item()}, step=step)
    wandb.log({f"Unpaired: \int f(y)": output_unpaired["int_potential"].item()}, step=step)
    wandb.log({f"Unpaired: \int\log Z": output_unpaired["int_log_Z"].item()}, step=step)
    wandb.log({f"Unpaired: -E(x, y)": output_unpaired["neg_energy_t"].item()}, step=step)
    wandb.log({f"Unpaired: c(x, y)": output_unpaired["cost_t"].item()}, step=step)
    wandb.log({f"Unpaired: f(y)": output_unpaired["potential_t"].item()}, step=step)
    wandb.log({f"Unpaired: noise": output_unpaired["noise"].item()}, step=step)

    D_opt_paired.zero_grad()
    X_paired, Y_paired = T_XY_sampler.sample(BATCH_SIZE)
    X_paired = X_paired.squeeze(1)
    Y_paired = Y_paired.squeeze(1)

    output_paired = model.compute_paired_loss(X_paired, Y_paired, compute_stats=True)
    D_loss_paired = output_paired["loss"]

    wandb.log({f"Paired: Loss": D_loss_paired.item()}, step=step)
    # wandb.log({f"Paired: -E(x, y)": output_paired["neg_energy_t"].item()}, step=step)
    # wandb.log({f"Paired: c(x, y)": output_paired["cost_t"].item()}, step=step)
    # wandb.log({f"Paired: f(y)": output_paired["potential_t"].item()}, step=step)
    # wandb.log({f"Paired: noise": output_paired["noise"].item()}, step=step)

    D_loss = D_loss_unpaired + D_loss_paired
    D_loss.backward()
    D_opt_paired.step()
    D_opt_unpaired.step()

    if train_config.ema_update:
        update_average(model_copy, model, 0.99)
        model = model_copy
    else:
        model = model

    wandb.log({f"Loss": D_loss}, step=step)
    # wandb.log(
    #     {f"Train paired loss": compute_loss(model, X_paired_train, Y_paired_train, X_paired_train, Y_paired_train)},
    #     step=step,
    # )
    # wandb.log(
    #     {f"Test paired loss": compute_loss(model, X_paired_test, Y_paired_test, X_paired_test, Y_paired_test)},
    #     step=step,
    # )
    # wandb.log(
    #     {f"Test unpaired loss": compute_loss(model, X_unpaired_test, Y_unpaired_test, X_paired_test, Y_paired_test)},
    #     step=step,
    # )

    if step % train_config.plot_every == 0:
        X_plot, Y_plot = T_XY_sampler.sample(10)
        X_plot = X_plot.squeeze(1)
        Y_plot = Y_plot.squeeze(1)
        Y_pred = model(X_plot)
        distr_dict = plot_images(X_plot, Y_pred, Y_plot, log=True)
        wandb.log(distr_dict)
        torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"model_{step}.pt"))

torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"model_{train_config.steps_to}.pt"))
torch.save(D_opt_paired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_paired_{train_config.steps_to}.pt"))
torch.save(D_opt_unpaired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{train_config.steps_to}.pt"))

wandb.finish()

wandb: Currently logged in as: muxaujl11110. Use `wandb login --relogin` to force relogin


  0%|                                                                                                                     | 0/2501 [00:00<?, ?it/s]/trinity/home/m.persiyanov/miniconda3/envs/text/lib/python3.11/site-packages/torch/autograd/graph.py:825: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
 14%|██████████████▌                                                                                          | 348/2501 [17:51<1:50:26,  3.08s/it]


KeyboardInterrupt: 

## Plotting

In [45]:
model.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"model_{1000}.pt"), map_location=device))

/tmp/ipykernel_2773152/2830823410.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"model_{1000}.pt"), map_loc

<All keys matched successfully>

In [46]:
X, Y = T_XY_sampler.sample(10)
X = X.squeeze(1)
Y = Y.squeeze(1)

In [47]:
Y_pred = model(X)

In [48]:
X.shape, Y.shape, Y_pred.shape

(torch.Size([10, 1, 32, 32]),
 torch.Size([10, 1, 32, 32]),
 torch.Size([10, 1, 32, 32]))

In [ ]:
plot_images(X, Y_pred, Y)